# Badminton singles analysis: Task 5 (TrackNetV3 vs BFMD shuttle track) on Google Colab

Runs TrackNetV3 on a 50 s clip of a broadcast video and compares its shuttle positions with the
BFMD shuttle track, with the same steps and settings as the local run in `REPORT.md`.

**Runtime:** GPU (Runtime > Change runtime type > **T4 GPU**).

**Google Drive layout expected** (`My Drive/badminton/`):
```
data/BFMD_data/annotations/<type>/<match>.json   BFMD annotations
videos/<match>.mp4                               the match video (here: China Open 2025 final)
task5/TrackNetV3_ckpts.zip                       created by this notebook (checkpoint cache)
outputs/task5/                                   written by this notebook
```

**Caveat:** BFMD's shuttle track is itself TrackNet output (unspecified version), so the
result is **agreement between two trackers, not accuracy**.

## 1. Settings and Google Drive

In [ ]:
import os, sys, json, glob, time, shutil, hashlib, subprocess, zipfile

if 'google.colab' in sys.modules:
    from google.colab import drive
    drive.mount('/content/drive')

# Paths (environment variables only override them for testing outside Colab)
DRIVE  = os.environ.get('BADMINTON_DRIVE', '/content/drive/MyDrive/badminton')
WORK   = os.environ.get('TASK5_WORK', '/content/task5')        # local disk: clip, TrackNetV3, predictions
REPO   = os.environ.get('TASK5_REPO', '/content/repo')
CONFIG = os.environ.get('TASK5_CONFIG', 'configs/colab.yaml')  # relative to REPO

GITHUB_URL = "https://github.com/bzaat/Badminton_Project.git"
BRANCH     = os.environ.get('TASK5_BRANCH', 'main')
MATCH      = 'Wang-Zheng-Xing-CHN-F'                            # China Open 2025, women's singles final

TRACKNET_URL    = 'https://github.com/qaz812345/TrackNetV3.git'
TRACKNET_COMMIT = '6eda442'
CKPT_DRIVE_ID   = '1CfzE87a0f6LhBp0kniSl1-89zaLCZ8cA'
CKPT_SHA256     = '8de68d34ea2457e368e22947f1309143d60a6a7f9ddccd31cd526d4c4571bcc4'
CKPT_CACHE      = f'{DRIVE}/task5/TrackNetV3_ckpts.zip'

def sh(cmd, cwd=None, env=None):
    """Run a command, stream its output, raise if it fails; returns the output text."""
    print('$', ' '.join(map(str, cmd)), flush=True)
    p = subprocess.Popen(list(map(str, cmd)), cwd=cwd, env=env, stdout=subprocess.PIPE,
                         stderr=subprocess.STDOUT, text=True, bufsize=1)
    out = []
    for line in p.stdout:
        out.append(line)
        if '\r' not in line:          # skip progress-bar redraws
            print(line, end='', flush=True)
    if p.wait():
        print(''.join(out[-30:]))
        raise RuntimeError(f'command failed with exit code {p.returncode}')
    return ''.join(out)

assert os.path.isdir(f'{DRIVE}/data/BFMD_data/annotations'), f'BFMD annotations not found under {DRIVE}/data'
os.makedirs(WORK, exist_ok=True)
print('DRIVE', DRIVE, '| WORK', WORK)

## 2. GPU check

In [ ]:
import torch
print('torch', torch.__version__, '| CUDA', torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError('No GPU. Runtime > Change runtime type > T4 GPU, then run all again '
                       '(on CPU TrackNetV3 would take hours).')
print(torch.cuda.get_device_name(0), f'{torch.cuda.get_device_properties(0).total_memory / 2**30:.1f} GiB')

## 3. Get the code (this project)

In [ ]:
os.chdir('/content' if os.path.isdir('/content') else WORK)   # never delete the directory we stand in
if os.path.exists(REPO):
    shutil.rmtree(REPO)
sh(['git', 'clone', '-q', '--branch', BRANCH, GITHUB_URL, REPO])
os.chdir(REPO)
assert os.path.exists(f'{REPO}/badminton_analysis/task5.py'), 'this checkout has no Task 5 code'
print(sh(['git', 'log', '--oneline', '-1'], cwd=REPO))

## 4. Copy this match's BFMD annotations to local disk
Task 5 only needs the one match, so only its files are copied (a few seconds instead of 1.8 GB).

In [ ]:
BFMD = f'{WORK}/BFMD_data'
shutil.rmtree(BFMD, ignore_errors=True)
n = 0
for src in sorted(glob.glob(f'{DRIVE}/data/BFMD_data/annotations/*/*{MATCH}*.json')):
    dst = f'{BFMD}/annotations/{os.path.basename(os.path.dirname(src))}/{os.path.basename(src)}'
    os.makedirs(os.path.dirname(dst), exist_ok=True)
    shutil.copy(src, dst); n += 1
types = sorted(os.listdir(f'{BFMD}/annotations'))
print(n, 'files:', types)
for t in ('metadata', 'hit_inferred', 'shot_type', 'pose', 'shuttle', 'player_bbox', 'court', 'court_perframe'):
    assert t in types, f'missing {t} annotation for {MATCH}: Drive upload incomplete?'
os.environ['BFMD_ROOT'] = BFMD           # the config reads BFMD_ROOT before its own bfmd_root path

## 5. TrackNetV3 (pinned commit) and its extra dependencies
TrackNetV3's `requirements.txt` pins torch 1.10 / numpy 1.22, which don't install on current
Python; it runs fine with Colab's libraries. `predict.py` additionally needs `parse` and
`pycocotools` (missing from its requirements); `gdown` fetches the checkpoints.

In [ ]:
import importlib.util
missing = [m for m in ('parse', 'pycocotools', 'gdown') if importlib.util.find_spec(m) is None]
if missing:
    sh([sys.executable, '-m', 'pip', 'install', '-q', *missing])
TN = f'{WORK}/TrackNetV3'
if not os.path.isdir(f'{TN}/.git'):
    sh(['git', 'clone', '-q', TRACKNET_URL, TN])
sh(['git', 'checkout', '-q', TRACKNET_COMMIT], cwd=TN)
print(sh(['git', 'log', '--oneline', '-1'], cwd=TN))

## 6. Checkpoints (126 MB, MIT licence), verified by SHA-256
Downloaded once from the TrackNetV3 authors' Google Drive link and cached in your Drive.
If the download fails (Drive quota), download `TrackNetV3_ckpts.zip` by hand from the
TrackNetV3 README and put it at `My Drive/badminton/task5/TrackNetV3_ckpts.zip`.

In [ ]:
def sha256(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for b in iter(lambda: f.read(1 << 20), b''):
            h.update(b)
    return h.hexdigest()

os.makedirs(os.path.dirname(CKPT_CACHE), exist_ok=True)
if not os.path.exists(CKPT_CACHE) or sha256(CKPT_CACHE) != CKPT_SHA256:
    import gdown
    tmp = f'{WORK}/TrackNetV3_ckpts.zip'
    gdown.download(id=CKPT_DRIVE_ID, output=tmp, quiet=False)
    got = sha256(tmp)
    assert got == CKPT_SHA256, f'checkpoint SHA-256 mismatch: {got}'
    shutil.copy(tmp, CKPT_CACHE)
print('SHA-256 OK:', sha256(CKPT_CACHE))
with zipfile.ZipFile(CKPT_CACHE) as z:
    z.extractall(TN)
for f in ('TrackNet_best.pt', 'InpaintNet_best.pt'):
    print(f, f'{os.path.getsize(f"{TN}/ckpts/{f}") / 2**20:.1f} MiB')

## 7. Cut the clip
`task5-clip` picks the first 3 consecutive clean rallies of one game (+1 s margin) and writes
them to their own .mp4 (clip frame i = video frame `clip_start` + i).

In [ ]:
videos = glob.glob(f'{DRIVE}/videos/*{MATCH}*.mp4')
assert len(videos) == 1, f'expected one video for {MATCH} in {DRIVE}/videos, found {videos}'
VIDEO = videos[0]
CLIP = f'{WORK}/clip.mp4'
out = sh([sys.executable, '-m', 'badminton_analysis', '--config', CONFIG, 'task5-clip',
          '--match', MATCH, '--video', VIDEO, '--out', CLIP], cwd=REPO)
clip = json.loads(out.strip().splitlines()[-1])
CLIP_START = clip['clip_start']
print(clip)
assert clip['frames'] == clip['clip_end'] - clip['clip_start'], 'video ended before the clip did'
if MATCH == 'Wang-Zheng-Xing-CHN-F':      # the local run in REPORT.md
    assert (clip['clip_start'], clip['frames']) == (14339, 1508), 'clip differs from the local run'

## 8. Run TrackNetV3
Same flags as the local run: `--large_video` streams the clip instead of loading it into RAM,
and `--max_sample_num 300` keeps the background-median step from running out of memory.

In [ ]:
PRED_DIR = f'{WORK}/pred_out'
shutil.rmtree(PRED_DIR, ignore_errors=True)
t = time.time()
sh([sys.executable, 'predict.py', '--video_file', CLIP,
    '--tracknet_file', 'ckpts/TrackNet_best.pt', '--inpaintnet_file', 'ckpts/InpaintNet_best.pt',
    '--save_dir', PRED_DIR, '--large_video', '--max_sample_num', '300'], cwd=TN)
PRED = f'{PRED_DIR}/clip_ball.csv'
assert os.path.exists(PRED), f'no prediction CSV in {PRED_DIR}: {os.listdir(PRED_DIR)}'
n_rows = sum(1 for _ in open(PRED)) - 1
print(f'TrackNetV3 took {time.time() - t:.0f} s for {n_rows} frames')
assert n_rows == clip['frames'], f'{n_rows} prediction rows for {clip["frames"]} clip frames'

## 9. Compare with the BFMD shuttle track

In [ ]:
out = sh([sys.executable, '-m', 'badminton_analysis', '--config', CONFIG, 'task5-eval', '--match', MATCH,
          '--pred', PRED, '--clip-start', CLIP_START, '--video', VIDEO], cwd=REPO)
sys.path.insert(0, REPO)
from badminton_analysis.config import load_config
OUT5 = load_config(os.path.join(REPO, CONFIG)).output_dir / 'task5'
shutil.copy(PRED, OUT5 / 'clip_ball.csv')          # keep the raw predictions with the results
res = json.load(open(OUT5 / 'metrics.json'))
print('results in', OUT5)

## 10. Results, and check against the local run
Local reference: RTX 3050 laptop GPU, torch 2.14 (REPORT.md). A different GPU, PyTorch and
OpenCV build (the clip is re-encoded) can move a few frames, so small differences are expected;
the frame-alignment check must still pick shift 0.

In [ ]:
from IPython.display import Markdown, Image, display
display(Markdown(open(OUT5 / 'report.md').read()))

REF = {'both_visible': (556, 15), 'only_bfmd': (52, 10), 'only_tracknetv3': (22, 10),
       'visibility_agreement': (0.883, 0.02), 'error_px_median': (2.2, 0.5),
       'within_5px': (0.730, 0.03), 'within_20px': (0.842, 0.03), 'best_frame_shift': (0, 0)}
rows, bad = [], []
for k, (ref, tol) in REF.items():
    v = res[k]; ok = v is not None and abs(v - ref) <= tol
    rows.append(f'| {k} | {v:.3f} | {ref} | ±{tol} | {"ok" if ok else "**DIFFERENT**"} |'
                if isinstance(v, float) else f'| {k} | {v} | {ref} | ±{tol} | {"ok" if ok else "**DIFFERENT**"} |')
    if not ok: bad.append(k)
display(Markdown('| metric | this run | local run | tolerance | |\n|---|---|---|---|---|\n' + '\n'.join(rows)))
print('All metrics within tolerance of the local run.' if not bad else f'Outside tolerance: {bad}')

for p in res.get('overlays', []):
    print(os.path.basename(p)); display(Image(p, width=900))